# Live Runner

## Overview

This notebook demonstrates `LiveRunner.run_once` in `live_runner` using one deterministic dry-run signal.
- Problem: a trading signal must pass each runtime boundary in the correct order before it can become an order.
- Approach: inject a deterministic market-data adapter, force a dry-run state store, and run one normalized signal through `LiveRunner`.
- Single Signal: It processes one buy signal without network access or exchange order submission.
- Runtime Result: It displays the simulated order and the position calculated from recorded executions.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory


In [ ]:
from IPython.display import display

from src.live_trading.config import load_live_trading_config
from src.live_trading.kraken_client import KrakenClient
from src.live_trading.live_runner import LiveRunner
from src.live_trading.order_manager import OrderManager
from src.live_trading.risk_manager import RiskManager


## Single Signal

This cell defines a deterministic exchange adapter for the dry-run example.
- `fetch_ticker` supplies a fixed public-market-price-shaped response.
- No private exchange operation is implemented, so the example cannot submit a live order.
- The temporary SQLite database isolates the demonstration from the configured Trading State Store.


In [ ]:
class DryRunExchange:
    """Provide the one public ticker operation needed by this example."""

    def fetch_ticker(self, symbol: str) -> dict[str, float]:
        """Return a deterministic last price for the configured symbol."""
        return {"last": 100.0}


config = load_live_trading_config()
if not config.dry_run:
    raise ValueError("Set KRAKEN_DRY_RUN=true before running this notebook example.")

signal = {"side": "buy", "amount": config.order_size}

with TemporaryDirectory() as temporary_directory:
    client = KrakenClient(config, exchange=DryRunExchange())
    order_manager = OrderManager(
        client=client,
        state_db_path=Path(temporary_directory) / "trading_state.db",
        dry_run=True,
    )
    risk_manager = RiskManager(config.max_order_size, config.max_position)
    runner = LiveRunner(config, client, order_manager, risk_manager)
    order = runner.run_once(signal)
    position = order_manager.get_position(config.symbol)

display({"signal": signal, "market_price": 100.0, "order": order})
display({"symbol": config.symbol, "net_position": position})
